# TopAneu A-N00-TYPE

TopAneu 2026 MRA `type_masks` 기반
Standard nnU-Net Type segmentation baseline 재현 Notebook.

Experiment:

`A-N00-TYPE`

Dataset:

`Dataset502_TopAneuMRA_Type`


## 1. Google Drive Mount


In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive"
)


## 2. 고정 경로

Location nnU-Net 실험과 동일하게
Python 3.12.13 `/content/venv312` 환경을 사용한다.


In [ ]:
from pathlib import Path

REPO = Path(
    "/content/model_type_baseline"
)

PREPROCESSED = Path(
    "/content/nnUNet_preprocessed"
)

RESULTS = Path(
    "/content/drive/MyDrive/Aneu_MRA/"
    "nnUNet_results_A_N00_TYPE"
)

PYTHON_BIN = Path(
    "/content/venv312/bin/python"
)

DATASET_NAME = (
    "Dataset502_TopAneuMRA_Type"
)

print(
    "REPO        :",
    REPO,
)

print(
    "PREPROCESSED:",
    PREPROCESSED,
)

print(
    "RESULTS     :",
    RESULTS,
)

print(
    "PYTHON      :",
    PYTHON_BIN,
)


## 3. Python 환경 확인


In [ ]:
import subprocess

if not PYTHON_BIN.is_file():

    raise FileNotFoundError(
        f"Python 3.12 venv 없음: {PYTHON_BIN}"
    )

subprocess.run(
    [
        str(PYTHON_BIN),
        "--version",
    ],
    check=True,
)


## 4. Dataset502 Preprocessed 확인

Dataset502 생성 및 preprocessing은
본학습 전에 별도 필수 단계로 수행한다.


In [ ]:
dataset_dir = (
    PREPROCESSED
    / DATASET_NAME
)

print(
    "Dataset502 preprocessed:",
    dataset_dir.is_dir(),
)

if not dataset_dir.is_dir():

    print(
        "Dataset502 preprocessing이 아직 필요합니다."
    )


## 5. A-N00-TYPE 학습 / Resume

Launcher는 checkpoint 존재 여부를 확인하여
NEW 또는 RESUME 모드를 자동 선택한다.


In [ ]:
import subprocess

launcher = (
    REPO
    / "scripts/training/"
    "run_a_n00_type.py"
)

command = [
    str(
        PYTHON_BIN
    ),

    str(
        launcher
    ),

    "--preprocessed",
    str(
        PREPROCESSED
    ),

    "--results",
    str(
        RESULTS
    ),

    "--python-bin",
    str(
        PYTHON_BIN
    ),
]

print(
    " ".join(
        command
    )
)

# Dataset502 preprocessing 완료 후 실행.
# subprocess.run(
#     command,
#     check=True,
# )


## 6. Final Validation 확인

본학습 및 Final Validation 완료 후 사용한다.


In [ ]:
fold_dir = (
    RESULTS
    / DATASET_NAME
    / (
        "nnUNetTrainerAneurysmTypeBaseline"
        "__nnUNetPlans__3d_fullres"
    )
    / "fold_0"
)

validator = (
    REPO
    / "scripts/validation/"
    "check_a_n00_type_final_validation.py"
)

validation_command = [
    str(
        PYTHON_BIN
    ),

    str(
        validator
    ),

    "--fold-dir",
    str(
        fold_dir
    ),
]

print(
    " ".join(
        validation_command
    )
)

# Final Validation 완료 후 실행.
# subprocess.run(
#     validation_command,
#     check=False,
# )


## 7. 학습 기준

- Standard nnU-Net PlainConvUNet
- Dataset502
- Background + Type 1/2/3 = 4 outputs
- Train / Validation / Test = 215 / 46 / 46
- Max epochs = 300
- Train iterations = 250
- Online validation iterations = 50
- Foreground oversampling = 0.33
- Best monitor = ema_fg_dice
- EarlyStopping = start 0 / patience 30 / min_delta 0.0001
- Strict Resume 적용
- Final Validation = checkpoint_best로 Validation 46명 1회
